# A-11 (수정판) — ViT 아키텍처 일반화: **ViT 자체에 대해 공격 생성**
이전 v1의 문제: (1) CIFAR ViT 헤드 미학습, (2) ImageNet 적대 예제가 ResNet에 대해 생성돼 ViT를
못 속임 → 예측특징이 전이 실패로 0.45. 본 v2는 **ImageNet ViT-B/16(정품 사전학습)에 대해
FGSM/PGD/C&W를 새로 생성**하여 공정 평가한다.
- HF-Energy(모델 불필요)는 ResNet/ViT 동일해야 함(백본 독립 입증)
- GaussianL1/PredL1/앙상블이 ViT-자체 공격을 잡는지 확인
출력 → paper v6 표 13.

In [ ]:
import os, sys, subprocess, pickle, io as _io
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

SEARCH_ROOTS=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
def _find(name, ftype='f', maxdepth=8):
    res=[]
    for root in SEARCH_ROOTS:
        if not os.path.exists(root): continue
        try:
            out=subprocess.run(['find',root,'-maxdepth',str(maxdepth),'-type',ftype,'-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)
CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
IMGNET_MEAN=[0.485,0.456,0.406];   IMGNET_STD=[0.229,0.224,0.225]
def make_preprocess(ds):
    m,s=(CIFAR_MEAN,CIFAR_STD) if 'CIFAR' in ds else (IMGNET_MEAN,IMGNET_STD)
    mean=torch.tensor(m).view(1,3,1,1); std=torch.tensor(s).view(1,3,1,1)
    return lambda x:(x/255.0-mean.to(x.device))/std.to(x.device)
def load_backbone(ds):
    if ds=='CIFAR-10':
        ck=(_find('resnet50_cifar10_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,10)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=10
    elif ds=='CIFAR-100':
        ck=(_find('resnet50_cifar100_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,100)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=100
    elif ds=='SVHN':
        ck=(_find('resnet50_svhn_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,10)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=10
    elif ds in ('TinyImageNet','Tiny ImageNet'):
        ck=(_find('resnet50_tinyimagenet_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,200)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=200
    else:
        m=models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2); nc=1000
    return m.to(device).eval(), nc
def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: out.setdefault('CIFAR-10',p)
        elif 'cifar100' in pl: out.setdefault('CIFAR-100',p)
        elif 'svhn' in pl: out.setdefault('SVHN',p)
        elif 'tiny' in pl: out.setdefault('TinyImageNet',p)
        elif 'imagenet' in pl and 'eps8' in pl: out.setdefault('ImageNet_eps8',p)
    return out
def gb(x,sigma):
    k=int(2*np.ceil(3*sigma)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=sigma)
def to224(img,mode='bicubic'):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224 or img.shape[-2]!=224:
        img=F.interpolate(img,size=(224,224),mode=mode,align_corners=False)
    return img.clamp(0,255)
def jpeg(img224,q):
    arr=img224.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    buf=_io.BytesIO(); _Image.fromarray(arr).save(buf,format='JPEG',quality=int(q)); buf.seek(0)
    return torch.from_numpy(np.array(_Image.open(buf).convert('RGB'))).float().permute(2,0,1).unsqueeze(0).to(img224.device)
def jpeg_batch(x,q=75):
    return torch.cat([jpeg(x[i:i+1],q) for i in range(x.shape[0])],0)
def median3(x):
    xx=F.pad(x,(1,1,1,1),mode='reflect'); p=xx.unfold(2,3,1).unfold(3,3,1)
    return p.contiguous().view(*p.shape[:4],9).median(dim=-1).values
def load_mixed(pkl_path, n_clean=500):
    with open(pkl_path,'rb') as f: mixed=pickle.load(f)
    clean=[im for (im,lb,atk) in mixed if atk=='clean']
    adv  =[(im,atk) for (im,lb,atk) in mixed if atk!='clean']
    rng=np.random.RandomState(SEED); idx=np.arange(len(clean)); rng.shuffle(idx)
    return [clean[i] for i in idx[:n_clean]], adv
print('header ready; device=',device)

In [ ]:
from torchvision.models import vit_b_16, ViT_B_16_Weights
vit=vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1).to(device).eval()
resnet,_=load_backbone('ImageNet')
pp=make_preprocess('ImageNet')
MX=find_mixed(); clean,_=load_mixed(MX['ImageNet_eps8'],n_clean=400)
imgs=[to224(im).to(device) for im in clean[:300]]
print('clean imgs:',len(imgs))

In [ ]:
# ViT 자체에 대한 공격 생성
def fgsm(x,y,model,eps=8.0):
    x=x.clone().requires_grad_(True)
    loss=F.cross_entropy(model(pp(x)),y); g=torch.autograd.grad(loss,x)[0]
    return (x+eps*g.sign()).clamp(0,255).detach()
def pgd(x,y,model,eps=8.0,a=2.0,steps=20):
    xa=(x+torch.empty_like(x).uniform_(-eps,eps)).clamp(0,255).detach()
    for _ in range(steps):
        xa.requires_grad_(True)
        loss=F.cross_entropy(model(pp(xa)),y); g=torch.autograd.grad(loss,xa)[0]
        xa=(xa+a*g.sign()).clamp(x-eps,x+eps).clamp(0,255).detach()
    return xa
def cw(x,y,model,c=1.0,steps=100,lr=0.01):
    xn=(x/255.).clamp(1e-4,1-1e-4); w=torch.atanh(xn*2-1).clone().requires_grad_(True)
    opt=torch.optim.Adam([w],lr=lr)
    for _ in range(steps):
        xa=(torch.tanh(w)+1)/2*255.; lo=model(pp(xa))
        zt=lo.gather(1,y.view(-1,1)).squeeze(1); oth=lo.clone(); oth.scatter_(1,y.view(-1,1),-1e9)
        loss=((xa-x)/255.).pow(2).flatten(1).sum(1)+c*torch.clamp(zt-oth.max(1).values,min=0.)
        opt.zero_grad(); loss.sum().backward(); opt.step()
    return ((torch.tanh(w)+1)/2*255.).detach()
# 정상 + ViT-적대 생성
X=torch.cat(imgs,0)
with torch.no_grad(): Yv=vit(pp(X)).argmax(1)
ADV={}
ADV['fgsm']=torch.cat([fgsm(X[i:i+1],Yv[i:i+1],vit) for i in range(len(X))],0)
ADV['pgd'] =torch.cat([pgd (X[i:i+1],Yv[i:i+1],vit) for i in range(len(X))],0)
ADV['cw']  =torch.cat([cw  (X[i:i+1],Yv[i:i+1],vit) for i in range(len(X))],0)
for k,v in ADV.items():
    with torch.no_grad(): asr=(vit(pp(v)).argmax(1)!=Yv).float().mean().item()
    print(f'ViT {k} ASR={asr:.2f}')

In [ ]:
def hfe_b(x): return ((x-gb(x,0.5)).abs().flatten(1).mean(1)/255.0).cpu().numpy()
def gl_b(x,model):
    with torch.no_grad():
        p0=F.softmax(model(pp(x)),1); p1=F.softmax(model(pp(gb(x,1.0))),1)
    return (p0-p1).abs().sum(1).cpu().numpy()
def predl1_b(x,model):
    with torch.no_grad():
        p0=F.softmax(model(pp(x)),1); sq=median3(jpeg_batch(x,75)).clamp(0,255)
        p1=F.softmax(model(pp(sq)),1)
    return (p0-p1).abs().sum(1).cpu().numpy()
def batched(fn,x,bs=32,**kw):
    return np.concatenate([fn(x[i:i+bs],**kw) for i in range(0,len(x),bs)])
# 정상 특징
Hc=batched(lambda z:hfe_b(z),X); Gc=batched(lambda z:gl_b(z,vit),X); Pc=batched(lambda z:predl1_b(z,vit),X)
HcR=batched(lambda z:gl_b(z,resnet),X)  # (참고) resnet 예측특징 비교용 — 생략 가능
def A(c,a):
    mu=c.mean();sd=c.std()+1e-8
    return roc_auc_score(np.r_[np.zeros(len(c)),np.ones(len(a))],np.r_[np.abs((c-mu)/sd),np.abs((a-mu)/sd)])
def ens(Hc,Gc,Pc,Ha,Ga,Pa):
    def z(c,a): m=c.mean();s=c.std()+1e-8; return np.abs((c-m)/s),np.abs((a-m)/s)
    zhc,zha=z(Hc,Ha); zgc,zga=z(Gc,Ga); zpc,zpa=z(Pc,Pa)
    nc=(zhc+zgc+zpc)/3; na=(zha+zga+zpa)/3
    return roc_auc_score(np.r_[np.zeros(len(nc)),np.ones(len(na))],np.r_[nc,na])
print(f"{'attack':<8}{'HF-Energy':>11}{'GaussL1':>10}{'PredL1':>10}{'Ensemble':>10}  (ViT-B/16, ImageNet)")
for k,v in ADV.items():
    Ha=batched(lambda z:hfe_b(z),v); Ga=batched(lambda z:gl_b(z,vit),v); Pa=batched(lambda z:predl1_b(z,vit),v)
    print(f"{k:<8}{A(Hc,Ha):>11.4f}{A(Gc,Ga):>10.4f}{A(Pc,Pa):>10.4f}{ens(Hc,Gc,Pc,Ha,Ga,Pa):>10.4f}")
print("\n핵심: HF-Energy AUC가 ResNet 결과(표4 ImageNet)와 유사하면 백본 독립 입증.")
print("      GaussL1/PredL1이 ViT-자체 cw/pgd를 잡으면(>0.7) 예측특징도 아키텍처 일반화.")